# Step 5: Pareto 调优（逐步 ignore → PPL 恢复曲线 → 拐点）

**目标**：跑通工业标准 Pareto 调优流程——(1) 全量化基线测 PPL；(2) 逐层敏感度排序（s1 产出）；(3) 按敏感度从高到低**逐步加 ignore**，每加一层**重新量化并测 PPL**，记录 (回退层数, PPL) 对；(4) 画 PPL 恢复曲线，找**Pareto 拐点**（PPL 快速恢复到接近 FP16 的最少回退层数）。这就是「最少回退 → 最大精度」的工程答案。

**对应 OUTLINE 课时**：3.5 调优流程（~50 分钟）。


## 学完应能讲清（学完本节应能口头回答）
1. Pareto 调优的四个标准步骤是什么？为什么「逐步加 ignore」要**每步重新量化**（不能只改 config 不重量化）？
2. PPL 恢复曲线长什么样？**Pareto 拐点**在曲线上是哪个位置？（PPL 从陡降快速变平缓的转折）
3. 为什么拐点 = Pareto 最优点？（拐点前每回退一层收益大、拐点后收益骤减——再回退只是浪费显存）
4. OUTLINE 说「每次回退都要重新量化」——具体哪一步会变？（GPTQ 的 Hessian 因 ignore 改变而变，缓存失效）
5. 为什么最终评估要 `limit` 放大或全量，而调优过程可以用 `limit=250`？（调优只要相对排序，小子集方差可接受；最终裁决要低方差）


In [ ]:
%%capture
import pathlib, os, re, json
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.gptq import GPTQModifier


In [ ]:
# Setup cell（双 env：模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT =", MODULE_ROOT, "| 0.5B @", TINY_MODEL_DIR.exists())


## 原理：Pareto 调优流程（OUTLINE 3.5）

```
  1. 全量化基线（无 ignore）测 PPL_baseline —— 通常是调优起点（掉点最狠）
  2. 敏感度排序（s1 产出，按 per-layer PPL 贡献降序）
  3. for k in 0,1,2,...:
       ignore = top-k 敏感层
       重新量化（oneshot）测 PPL_k           ← 每步都重新量化（Hessian 变）
       记录 (k, PPL_k)
  4. 画 (k, PPL_k) 曲线 → 找拐点 = Pareto 最优 k*
     （PPL 快速恢复到接近 FP16 的最少 k）
```

**关键易错点**（OUTLINE 3.5）：
- **每次回退都要重新量化**：GPTQ 的 Hessian 依赖校准数据流经的网络结构。ignore 掉一层后，该层不量化、误差特性变了，下游层收到的校准激活也变了 → Hessian 必须重算。**不能只改 config.json 的 ignore 字段、复用旧的量化权重**——那会得到错误的 PPL。
- `limit=250` 调优足够（相对排序），但**最终评估要全量**（OUTLINE：小子集方差大）。

**Pareto 拐点**：曲线上 PPL「由陡降转平缓」的膝盖处。拐点前回退收益大、拐点后收益骤减——拐点就是「最少回退→最大精度」的 sweet spot。


### 端到端：Pareto 在 layer fallback 全流程的位置

Pareto = 把 s1（敏感度）+ s3（ignore 语法）+ s4（mixed-precision）**串成自动化调优循环**。s1-s4 是零件，s5 是引擎：循环驱动「加 ignore→重量化→测 PPL」，机器跑完全程，人只看曲线定 k*。这是整个 M3 的工程产物——一张 PPL 恢复曲线 + 一个 Pareto 拐点 k*。


## 亲手摸一摸：一条 PPL 恢复曲线 + 拐点

先用合成数据画一条典型 PPL 恢复曲线，直观看到「拐点」长什么样——陡降后转平的那个膝盖。


In [ ]:
## 摸一摸：合成 PPL 恢复曲线 + 拐点
# 模拟：基线全量化 PPL=12，FP16 目标 PPL=6，逐层 ignore 恢复
ks = list(range(0, 11))
# 前 3 层收益大（陡降），之后收益骤减（平缓）——典型 Pareto 形状
ppls = [12.0, 10.2, 8.5, 7.2, 6.9, 6.7, 6.6, 6.55, 6.52, 6.51, 6.5]
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(ks, ppls, 'o-', label='PPL (stepwise ignore)')
ax.axhline(6.0, ls='--', color='green', label='FP16 target PPL')
ax.axvline(3, ls=':', color='red', label='Pareto knee k*=3')
ax.set_xlabel('Num fallback layers k'); ax.set_ylabel('PPL'); ax.set_title('PPL recovery curve (synthetic demo)')
ax.legend(); fig.tight_layout(); plt.show()
print("拐点 k*=3：再往前 PPL 还在 7+，再往后收益骤减（6.9→6.5 花了 7 层）")


## 本步填空

1. **`pareto_step(sensitive_rank, k, fp16_ppl, baseline_ppl)`** —— 模拟调优循环的单步：给定敏感度排序、回退层数 k，返回该步的 (k, ppl, recovered_ratio)。用解析模型模拟「前 k 层回退后的 PPL」（指数恢复趋近 FP16），驱动循环。**为什么这么设计（填前先想）**：真重量化每步要数分钟，L1/L2 不可能真跑；用解析模型（PPL 按 k 指数趋近 FP16）模拟调优曲线的**形状**，让算法逻辑（循环、记录、排序）可测，L3 再换真重量化。
2. **`find_pareto_knee(curve, fp16_ppl)`**（判断型）—— 给定 (k, PPL) 曲线，找 Pareto 拐点：用「边际收益」法，找 PPL 单步降幅首次低于阈值的 k。**为什么这么设计**：拐点的数学定义就是「边际收益骤减」——用相邻步 PPL 差分，找差分从大变小的转折点。


In [ ]:
def pareto_step(sensitive_rank, k, fp16_ppl, baseline_ppl, decay=0.55):
    """模拟 Pareto 调优的第 k 步（回退 top-k 敏感层后的 PPL）。
    返回 (k, ppl, recovered_ratio)。

    解析模型：PPL(k) = fp16_ppl + (baseline_ppl - fp16_ppl) * decay^k
    —— 随 k 增大（0 ≤ k < N），PPL 指数趋近 FP16（前几层收益大、后骤减，符合真实 Pareto 形状）。
    recovered_ratio = (baseline - ppl) / (baseline - fp16) ∈ [0,1]：恢复了多大比例的精度。

    边界（重要）：k == N（len(sensitive_rank)，所有敏感层全回退）时，模型等价于全 FP16，
    PPL 精确等于 fp16、recovered = 1.0——指数衰减只能渐近趋近、到不了，需特判精确命中。

    为什么这么设计（填前先想）：真重量化（oneshot）每步数分钟，L1/L2 跑不了循环；
    用解析模型模拟曲线形状，让循环/排序/拐点逻辑可测。decay<1 保证单调下降趋近 fp16。
    """
    # TODO: 1) k clamp 到 [0, len(sensitive_rank)]。
    #       2) gap = baseline_ppl - fp16_ppl（全量化掉的精度总量）。gap≈0 时直接 recovered=1。
    #       3) 若 k == len(sensitive_rank)（全回退）：ppl=fp16_ppl、recovered=1.0（精确命中，见 docstring 边界）。
    #       4) 否则 ppl = fp16_ppl + gap * (decay ** k)；recovered = (baseline_ppl - ppl) / gap。
    #       5) 返回 (k, ppl, recovered)。
    raise NotImplementedError

In [ ]:
def find_pareto_knee(curve, min_drop_frac=0.10):
    """判断型：给定 (k, PPL) 曲线（list of (k, ppl)），找 Pareto 拐点 k*。
    用「边际收益」法：相邻步 PPL 降幅 Δ_i = ppl[i-1] - ppl[i]；
    拐点 = 第一个 Δ_i < min_drop_frac * 初始降幅 Δ_0 的步（边际收益骤减）。
    返回该步的 k（最少回退层数）。

    为什么这么设计（填前先想）：拐点的数学定义就是「边际收益骤减」。
    第一层 ignore 通常降最多（Δ_0 最大）；之后每层降幅递减。
    当降幅跌到不足初始的 min_drop_frac，说明再回退收益微薄——这就是拐点。
    """
    # TODO: 1) 若 len(curve) < 2 返回 curve[-1] 的 k（没法算差分）。
    #       2) drops[i] = curve[i-1][1] - curve[i][1] for i in 1..n-1。
    #       3) initial_drop = drops[1]（第一个降幅，k=0->1）。
    #       4) 找第一个 drops[i] < min_drop_frac * initial_drop 的 i，返回 curve[i][0]（k）。
    #       5) 若全程没跌破阈值，返回 curve[-1][0]（最后一个 k，说明持续高收益）。
    raise NotImplementedError


In [ ]:
def test_pareto_step_monotone_decreasing_to_fp16():
    rank = [f"layer_{i}" for i in range(10)]
    fp16, base = 6.0, 12.0
    prev = base
    for k in range(11):
        k_, ppl, rec = pareto_step(rank, k, fp16, base)
        assert k_ == min(k, 10)
        assert ppl <= prev + 1e-9, f"PPL 应随 k 单调不增: k={k} ppl={ppl} prev={prev}"
        assert ppl >= fp16 - 1e-6, "PPL 不应低于 FP16"
        assert 0.0 - 1e-6 <= rec <= 1.0 + 1e-6
        prev = ppl

def test_pareto_step_endpoints():
    rank = [f"layer_{i}" for i in range(5)]
    k0, ppl0, rec0 = pareto_step(rank, 0, 6.0, 12.0)
    assert abs(ppl0 - 12.0) < 1e-6 and abs(rec0) < 1e-6, "k=0 应等于基线"
    kN, pplN, recN = pareto_step(rank, 5, 6.0, 12.0)
    assert abs(recN - 1.0) < 1e-6, "k=N（全回退）应 recovered=1"

def test_find_pareto_knee_finds_turning_point():
    # 陡降 3 步后骤减：拐点应在 k=3 附近
    curve = [(0, 12.0), (1, 10.0), (2, 8.5), (3, 7.5),
             (4, 7.2), (5, 7.05), (6, 7.0)]  # drops: 2,1.5,1,0.3,0.15,0.05
    knee = find_pareto_knee(curve, min_drop_frac=0.15)
    # 初始降幅 2.0，阈值 0.3；drops: 2,1.5,1,0.3,0.15,0.05 —— 0.3 不 < 0.3，0.15<0.3 在 k=4
    assert knee == 4, f"期望拐点 k=4（边际收益首次跌破 15%），得 {knee}"

def test_find_pareto_knee_no_drop_returns_zero():
    curve = [(0, 10.0), (1, 10.0), (2, 10.0)]  # 完全没收益
    assert find_pareto_knee(curve) == 0

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"


## L2（tiny，CPU）：合成 Pareto 调优循环 + 画曲线找拐点

用 `pareto_step` 跑完整调优循环（模拟 10 步），画 PPL 恢复曲线，用 `find_pareto_knee` 找拐点。验证循环逻辑 + 拐点算法在典型曲线上给出合理 k*。


In [ ]:
## L2：合成 Pareto 调优循环
sensitive_rank = [f"model.layers.{i}.mlp.down_proj" for i in range(10)]  # s1 假想的排序
fp16_ppl, baseline_ppl = 6.0, 12.0

curve = []
for k in range(len(sensitive_rank) + 1):
    k_, ppl, rec = pareto_step(sensitive_rank, k, fp16_ppl, baseline_ppl)
    curve.append((k_, ppl))
print("Pareto 调优曲线 (k, PPL)：")
for k_, ppl in curve:
    print(f"  k={k_:2d}  PPL={ppl:.3f}")

knee = find_pareto_knee(curve, min_drop_frac=0.15)
print(f"\nPareto 拐点 k* = {knee}（最少回退层数 → 最大精度恢复）")

fig, ax = plt.subplots(figsize=(7,3.5))
xs = [c[0] for c in curve]; ys = [c[1] for c in curve]
ax.plot(xs, ys, 'o-'); ax.axhline(fp16_ppl, ls='--', color='green', label='FP16 target')
ax.axvline(knee, ls=':', color='red', label=f'Knee k*={knee}')
ax.set_xlabel('Num fallback layers k'); ax.set_ylabel('PPL'); ax.set_title('L2 Synthetic Pareto recovery curve')
ax.legend(); fig.tight_layout(); plt.show()

assert 1 <= knee <= len(sensitive_rank), "拐点应在合理范围"
print("\nL2 通过：调优循环 + 拐点算法在合成曲线上给出合理 k*。")


## L3（H200，GPU 守卫）：真 7B 逐步 ignore 重量化 Pareto

在 7B 上跑**真** Pareto 调优：用 s1 的敏感度排序，逐步（k=0,2,4,6）加 ignore、每步重新量化测 PPL，画真恢复曲线找拐点。这是 M3 的核心实证产物——OUTLINE 验收标准 5 要求 s5 的 L3 跑通。


In [ ]:
import torch, os
def run_l3_pareto():
    from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM
    calib = load_dataset("wikitext", "wikitext-2-raw-v1", split="train").shuffle(seed=0)["text"][:128]

    rank_path = OUT_ROOT / "s1_sensitive_rank.json"
    rank = json.loads(rank_path.read_text()) if rank_path.exists() else []
    if not rank:
        print("无 s1 敏感度排序（先跑 s1 L3），L3 跳过"); return
    sensitive = [r[0] for r in rank if r[0] != "lm_head"]

    def ppl_of(model_dir):
        m = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16, device_map="auto")
        # 简化：用固定文本算 PPL（生产应用 wikitext 全集）
        tok = AutoTokenizer.from_pretrained(model_dir)
        ids = tok("The future of AI depends on efficient inference at scale.", return_tensors="pt").input_ids.to(m.device)
        with torch.no_grad(): logits = m(ids).logits[0]
        loss = torch.nn.functional.cross_entropy(logits[:-1], ids[0,1:])
        return float(torch.exp(loss).item())

    curve = []
    for k in [0, 2, 4, 6, 8]:
        ignore = ["lm_head"] + [s for s in sensitive[:k]]
        recipe = [SmoothQuantModifier(smoothing_strength=0.8),
                  GPTQModifier(targets="Linear", scheme="W8A8", ignore=ignore)]
        out = OUT_ROOT / f"s5_pareto_k{k}"
        oneshot(model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), recipe=recipe,
                dataset=calib, num_calibration_samples=128, output_dir=str(out))
        ppl = ppl_of(out)
        curve.append((k, ppl)); print(f"  k={k} ignore={len(ignore)}层 -> PPL={ppl:.3f}")

    json.dump({"curve": curve, "sensitive_rank": sensitive[:8]},
              open(OUT_ROOT / "s5_pareto_curve.json", "w"), indent=2)
    knee = find_pareto_knee(curve)
    print(f"\n真 7B Pareto 拐点 k* = {knee}")

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_pareto()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 调优/拐点逻辑）")


## 产物检查


In [ ]:
import json
p = OUT_ROOT / "s5_pareto_curve.json"
if p.exists():
    d = json.loads(p.read_text())
    curve = d["curve"]
    print("真 7B Pareto 曲线：")
    for k, ppl in curve: print(f"  k={k}  PPL={ppl:.3f}")
    knee = find_pareto_knee(curve)
    print(f"拐点 k* = {knee}")
    fig, ax = plt.subplots(figsize=(7,3.5))
    ax.plot([c[0] for c in curve], [c[1] for c in curve], 'o-')
    ax.axvline(knee, ls=':', color='red', label=f'Knee k*={knee}')
    ax.set_xlabel('Num fallback layers k'); ax.set_ylabel('PPL'); ax.set_title('7B Pareto recovery curve'); ax.legend()
    fig.tight_layout(); plt.show()
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")
